# 02 Data preparation — corrected copy

This notebook is the maintained, runnable preparation walkthrough. The original `02_data_preparation.ipynb` is preserved exactly for side-by-side comparison. This copy uses the maintained `src` modules, frozen manifest, and canonical paths; it does not rewrite documentation or source modules.

## Source, target, and frozen assignments

The canonical source is `data/raw/heart_2022_with_nans.zip`. Source row order defines `row_id`. Exact matches across all original fields share `duplicate_group` and stay in one split. `heart_disease` is 1 if HadHeartAttack or HadAngina is Yes, 0 only if both are No, and otherwise unknown. Unknown targets are recorded and excluded. The two target-source fields are excluded from predictor exports. The frozen manifest is validated, never regenerated by this notebook.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

candidates = (Path.cwd(), *Path.cwd().parents)
ROOT = next((p for p in candidates if (p / "src" / "prepare_data.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the repository root or its notebooks directory")
sys.path.insert(0, str(ROOT))
from src.prepare_data import prepare_data, SOURCE, MANIFEST_DIR, OUTPUT_DIR, AUDIT_DIR
print("Repository:", ROOT)
print("Source:", SOURCE)
print("Frozen manifest:", MANIFEST_DIR / "split_manifest.csv")

Repository: C:\Users\PC\Documents\GitHub\DSAI5101-Project
Source: C:\Users\PC\Documents\GitHub\DSAI5101-Project\data\raw\heart_2022_with_nans.zip
Frozen manifest: C:\Users\PC\Documents\GitHub\DSAI5101-Project\data\manifests\split_manifest.csv


In [2]:
splits = prepare_data(check_seed_42=True)
assert {name: len(frame) for name, frame in splits.items()} == {"train": 308080, "validation": 66016, "test": 66015}
assert all(len(frame.columns) == 41 for frame in splits.values())
assert all("HadHeartAttack" not in frame and "HadAngina" not in frame for frame in splits.values())
display(pd.read_csv(AUDIT_DIR / "split_summary.csv"))

Seed-42 duplicate-group split matches the frozen manifest.


train: 308,080 raw-valued rows -> C:\Users\PC\Documents\GitHub\DSAI5101-Project\data\processed\training.csv


validation: 66,016 raw-valued rows -> C:\Users\PC\Documents\GitHub\DSAI5101-Project\data\processed\validation.csv


test: 66,015 raw-valued rows -> C:\Users\PC\Documents\GitHub\DSAI5101-Project\data\processed\test.csv


Unknown-target exclusions: 5,021; frozen artifacts unchanged.


,split,rows,positive_cases,negative_cases
0,train,308080,27826,280254
1,validation,66016,5963,60053
2,test,66015,5962,60053


## Raw-valued handoff and audits

`data/processed/training.csv` is Member 2’s input for EDA and observed-value associations. Validation is reserved for model decisions and test for final evaluation. All 38 original predictor values and missing positions remain intact. `heart_disease` is saved as the target but separated as `y` for modeling; `row_id` and `duplicate_group` are metadata, not predictors. Training missingness and numeric ranges are audited below. Flags prompt review and never delete rows. SleepHours uses the dictionary’s 1–24 range.

In [3]:
training = splits["train"]
X = training.drop(columns=["heart_disease", "row_id", "duplicate_group"])
y = training["heart_disease"]
print("Training rows with any missing predictor:", int(X.isna().any(axis=1).sum()))
print("Training rows with missing BMI:", int(X.BMI.isna().sum()))
display(pd.read_csv(AUDIT_DIR / "training_missing_summary.csv"))
display(pd.read_csv(AUDIT_DIR / "training_range_flags.csv"))

Training rows with any missing predictor: 135348
Training rows with missing BMI: 33328


,column,missing_count,missing_pct
0,State,0,0.000000
1,Sex,0,0.000000
2,GeneralHealth,745,0.241820
3,PhysicalHealthDays,7278,2.362373
4,MentalHealthDays,6042,1.961179
5,LastCheckupTime,5589,1.814139
6,PhysicalActivities,683,0.221696
7,SleepHours,3525,1.144183
8,RemovedTeeth,7599,2.466567
9,HadStroke,697,0.226240


,column,review_flag_count,lower_bound,upper_bound
0,PhysicalHealthDays,0,0,30.0
1,MentalHealthDays,0,0,30.0
2,SleepHours,0,1,24.0
3,HeightInMeters,0,0,NaN
4,WeightInKilograms,0,0,NaN
5,BMI,0,0,NaN


## Modeling handoff

`src.preprocessing.make_preprocessor(training)` returns a new unfitted transformer. Fit it inside an estimator `Pipeline` in each cross-validation training fold. The handoff CSVs are never imputed or encoded, and this notebook does not create a preprocessed cross-validation dataset. The older imputed archive is historical and must not be used for EDA or cross-validation.